# 2.5 Your turn: your own chat

Everything through [02.4](02.4-counting-and-averages.ipynb) ran on showcase data, chosen so
the pattern was certain to be there. This notebook has one worked example, to prove the pipes
are connected, and then it is your data and your question.

Your own chat has far fewer people than the IRC corpus does, which makes 02.4's trap sharper
rather than softer — a channel of five people has no room for "the eight regulars" to hide
in, and every person you drop from a comparison moves the answer more, not less.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import py3langid as langid
from goad_toolkit.datatransforms import CountValues, GroupAgg, Pipeline, TimeFeatures, TransformBase
from goad_toolkit.visualizer import PlotSettings

from scripts.plots import BarPlot
from wa_analyzer.data import load_own_chat

## 2.5.1 Loading your data

`load_own_chat()` raises when there is no export yet — on purpose. Everything below is about
*your* data; without it there is nothing here to do, and a notebook that quietly runs on
nothing teaches nothing. So if the next cell errors, that is not the notebook breaking — it
is the notebook telling you to go back to [01.3](../lesson1/01.3-your-own-chat.ipynb), run
the preprocessor, and point `current` in `config.toml` at the file it writes.

In [ ]:
def canonical_author(name: str) -> str:
    """Louis appears under three names in the export (a contact rename mid-chat);
    group them into one author before any per-person comparison."""
    if "louis" in str(name).lower():
        return "Louis"
    return name


own = load_own_chat()
own["author"] = own["author"].apply(canonical_author)
own["length"] = own.message.str.len()
own.head()

## 2.5.2 What's already loaded

`BarPlot` — the class 02.2 derived — is imported above from `scripts/plots.py`, ready to use.
`BarPlotWithError`, 02.4's error-bar version, lives in the same file if the comparison you
pick needs one (`from scripts.plots import BarPlotWithError`). One worked example first, so
you know the pipes connect before you start changing things:

In [ ]:
counts = Pipeline().add(CountValues, column="author").apply(own)

base_settings = PlotSettings(
    figsize=(8, 4),
    title="Messages per person",
    xlabel="messages",
    ylabel="",
)
BarPlot(base_settings).plot(data=counts, x="n", y="author", color="#cccccc")

## 2.5.3 Brainstorming with `goad`

This repo has the `goad` MCP server connected — the same one behind
[CLAUDE.md](../../CLAUDE.md)'s coaching policy. Three tools worth knowing about here, all on
the assistant side of this notebook rather than in it:

- **`goad_analysis_checklist`** walks the six-stage method — the question, the data, the
  shape, the encoding, the critique, the verification — one stage at a time, and will not
  move to the next until you've answered the current one in your own words. Ask your
  assistant to run it for a comparison you want to make in your own chat, and expect to be
  interviewed rather than handed a finding.
- **`goad_search`** / **`goad_list_concepts`** / **`goad_get_concept`** look up a concept
  directly — "which chart family fits a question about who talks to whom", say, or the
  Simpson's-paradox chapter again if 02.3 is worth a second read against your own data.
- **`goad_critique_visual`** runs the four-part visual-critique checklist once you have a
  chart to defend — the same "grey first, is the finding written on the plot, what could you
  delete" questions 02.2 and 02.3 used, put to *your* chart instead of the showcase ones.

The MCP is the coaching layer over `goad_toolkit`, but the toolkit is also its own repo:
[github.com/raoulg/goad_toolkit](https://github.com/raoulg/goad_toolkit). The `docs/`
folder there holds the same ten chapters the MCP serves — pipelines, plot composition, the
five families, visual critique — readable straight from GitHub, with runnable examples in
the source. When you want to browse rather than be interviewed, that is the place.

> **Your turn.**
>
> 1. Make one bar chart that answers a question you actually have about your chat. Order it,
>    grey it, and colour the one bar your title is about — 02.2's guidelines, applied to data
>    nobody chose for you.
> 2. Pick a comparison between two people or two groups. Compute it counting messages, then
>    counting people, the way 02.4 did. Do they agree?
> 3. Before you trust either chart: how many *people* does the claim actually rest on? If the
>    honest answer is two or three, that is worth knowing before the chart is.

In [ ]:
# Your turn: does the language Louis and I speak to each other shift after his move to NL,
# and does it shift the same way for both of us? Agreed with the `goad` coach (stages 1-4,
# see analysis-log.md) before writing any of this.

MOVE_DATE = pd.Timestamp("2024-11-01")


class LanguageFeature(TransformBase):
    """Detect the language of a text column with py3langid.

    Messages under `min_chars` are marked "unknown" -- not because the detector was
    unsure, but because a message that short (an "Ok", a bare emoji) isn't really
    evidence of practising a language either way.
    """

    def transform(
        self,
        data: pd.DataFrame,
        column: str,
        languages: list[str],
        feature: str = "language",
        min_chars: int = 3,
    ) -> pd.DataFrame:
        langid.set_languages(languages)
        text = data[column].fillna("")

        def detect(msg: str) -> str:
            if len(msg.strip()) < min_chars:
                return "unknown"
            return langid.classify(msg)[0]

        data[feature] = text.map(detect)
        return data


own_pipeline = Pipeline()
own_pipeline.add(TimeFeatures, column="timestamp")
own_pipeline.add(LanguageFeature, column="message", languages=["en", "nl", "fr"], feature="language")
own_lang = own_pipeline.apply(own)

own_lang["timestamp_naive"] = pd.to_datetime(own_lang["timestamp"], utc=True).dt.tz_convert(None)
own_lang["period"] = np.where(own_lang["timestamp_naive"] < MOVE_DATE, "before move", "after move")

# "unknown" (< 3 characters) dropped from every percentage below: too short to be
# either evidence of speaking a language or evidence of not speaking one.
lang_known = own_lang[own_lang["language"] != "unknown"]

PERIOD_ORDER = ["before move", "after move"]
LANGUAGE_ORDER = ["en", "nl", "fr"]
PERIOD_COLORS = {"before move": "#cccccc", "after move": "#D55E00"}

lang_known[["author", "period", "language"]].head()

### Plot 1 — the couple, one axes instead of two panels

The original ("Praat Nederlands met mij", `roos_01.4`) put before/after in two side-by-side
panels. The teacher's `preattentive-gestalt` comment was about exactly this: two panels
means comparing 2x3 things, less direct than putting the bars next to each other. Same data,
one axes, `hue="period"` instead of a second panel.

In [ ]:
couple_pct = (
    lang_known.groupby(["period", "language"])
    .size()
    .rename("n")
    .reset_index()
)
couple_pct["pct"] = couple_pct.groupby("period")["n"].transform(lambda s: 100 * s / s.sum())

plot1_settings = PlotSettings(
    figsize=(8, 4.5),
    title="Praat Nederlands met mij",
    xlabel="language",
    ylabel="% of classified messages",
)
fig, ax = BarPlot(plot1_settings).plot(
    data=couple_pct,
    x="language",
    y="pct",
    hue="period",
    order=LANGUAGE_ORDER,
    hue_order=PERIOD_ORDER,
    palette=PERIOD_COLORS,
)
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f%%", padding=2, fontsize=9)
ax.set_ylim(0, 100)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="")

n_before = int((lang_known["period"] == "before move").sum())
n_after = int((lang_known["period"] == "after move").sum())
fig.text(
    0.5, -0.05,
    f"Based on {n_before:,} classified messages before the move, {n_after:,} after "
    "(messages under 3 characters excluded -- too short to reflect a language choice either way).",
    ha="center", fontsize=8.5, color="dimgray",
)
plt.show()

### Plot 2 — the layer the teacher asked for: who leans toward which language

Plot 1 only shows the couple combined. The `verhaallijn` comment asked "who speaks which
language to whom" — this plot answers it directly: for each language and period, Louis's
share of *his own* messages minus Roos's share of *her own* messages. Zero means no gap;
above zero means Louis leans toward that language more than Roos does, in that period.

In [ ]:
person_period_pct = (
    lang_known.groupby(["author", "period", "language"])
    .size()
    .rename("n")
    .reset_index()
)
person_period_pct["pct"] = person_period_pct.groupby(["author", "period"])["n"].transform(
    lambda s: 100 * s / s.sum()
)

wide = person_period_pct.pivot_table(
    index=["period", "language"], columns="author", values="pct", fill_value=0
)
diff = (wide["Louis"] - wide["Roos"]).rename("diff").reset_index()

plot2_settings = PlotSettings(
    figsize=(8, 4.5),
    title="Who leans toward which language?",
    xlabel="language",
    ylabel="Louis's % minus Roos's % (percentage points)",
)
fig, ax = BarPlot(plot2_settings).plot(
    data=diff,
    x="language",
    y="diff",
    hue="period",
    order=LANGUAGE_ORDER,
    hue_order=PERIOD_ORDER,
    palette=PERIOD_COLORS,
)
ax.axhline(0, color="black", linewidth=0.8)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="")

counts = lang_known.groupby(["author", "period"]).size()
fig.text(
    0.5, -0.07,
    f"Louis: {counts[('Louis', 'before move')]:,} classified messages before the move, "
    f"{counts[('Louis', 'after move')]:,} after. "
    f"Roos: {counts[('Roos', 'before move')]:,} before, {counts[('Roos', 'after move')]:,} after.",
    ha="center", fontsize=8.5, color="dimgray",
)
plt.show()

### Can we trust this? — draft text, edit before it goes in the report

*(This is a first draft for you to check and rewrite in your own words — not final report
text.)*

Both charts drop messages shorter than 3 characters before computing any percentage —
`py3langid` needs at least a few characters to guess a language at all, and a message that
short ("Ok", a single emoji) isn't real evidence of practising a language either way, so
dropping it is a definition choice, not a data-quality compromise.

That said, the drop rate isn't even: Louis's short-message rate roughly **triples** after
the move (1.0% → 3.0%), while mine barely moves (1.05% → 1.15%). Checking what those short
messages actually are: about 45% are "Ok"/"ok" (an English filler, not Dutch), roughly a
third are bare emoji, and the rest are a mix — but a real minority, "Ja"/"ik", are genuine
Dutch words that the 3-character cutoff throws away. So the cutoff most likely **understates**
Louis's Dutch after the move slightly, rather than overstating it.

Both charts are built on thousands of classified messages per person and period (smallest
group: Louis, after the move, still several thousand) — plenty for the percentages
themselves to be stable. The real limit is n=2: every comparison here describes this one
relationship, not people in general.

## 2.5.4 Reflect

- Did counting people change the answer from counting messages? If it did, were you
  surprised, or had you already half-expected it?
- Is there a variable — active period, group role, which sub-conversation someone is in —
  that might split your comparison the way department split Berkeley's admissions in 02.3?
  You don't have to check it now. Just: does one come to mind?
- Look at the chart you made in 2.5.3. If a stranger saw only the chart, thirty seconds, no
  narration from you — would they land on the finding you actually have? If you're not sure,
  `goad_critique_visual` exists for exactly this question, and it's a fair thing to ask your
  assistant to run right now.
- What would it take to change your mind about the comparison you just made? If nothing
  would, it might not be a finding yet.

---

**Where this goes next.** Lesson 3 takes the same comparisons and puts time on the x-axis,
where a new failure appears: the difference between a pattern and the way you smoothed it.